# Session 2 — Your turn: The transformer block

Count the parameters of a block from its shapes, and check the property that makes rotary position embeddings work: the score between a rotated query and key depends only on how far apart they are.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np
from aij_check import check, close, summary

## Task 1: Count a block's parameters

For a pre-norm block with model width `d`, `h` heads (no change in total size) and an MLP hidden width `4 * d`, write `block_params(d)` that returns a dictionary with:
`attention` (the combined query/key/value matrix `d x 3d` plus its bias `3d`, and the output matrix `d x d` plus its bias `d`), `mlp` (two matrices `d x 4d` and `4d x d` with their biases) and `norms` (two layer norms, each with a scale and a shift of size `d`), and `total`.

In [2]:
def block_params(d):
    attention = d * 3 * d + 3 * d + d * d + d
    mlp = d * 4 * d + 4 * d + 4 * d * d + d
    norms = 2 * 2 * d
    return {"attention": attention, "mlp": mlp, "norms": norms, "total": attention + mlp + norms}

In [3]:
gpt2 = block_params(768)
check("GPT-2's attention has 2,362,368 parameters", lambda: gpt2["attention"] == 2_362_368)
check("GPT-2's MLP has 4,722,432 parameters", lambda: gpt2["mlp"] == 4_722_432)
check("the MLP has about twice the parameters of the attention", lambda: 1.9 < gpt2["mlp"] / gpt2["attention"] < 2.1)
check("twelve such blocks hold about 85 million parameters", lambda: 84e6 < 12 * gpt2["total"] < 86e6)
check("the total is the sum of the parts", lambda: gpt2["total"] == gpt2["attention"] + gpt2["mlp"] + gpt2["norms"])
summary()

PASS  GPT-2's attention has 2,362,368 parameters
PASS  GPT-2's MLP has 4,722,432 parameters
PASS  the MLP has about twice the parameters of the attention
PASS  twelve such blocks hold about 85 million parameters
PASS  the total is the sum of the parts

5 of 5 checks passed.  Well done.


**Why it works.** Attention costs about 4d^2 parameters and the MLP 8d^2, so two thirds of a block's weights are in the MLP. Twelve blocks give 85M of GPT-2's 124M; the rest is the token and position tables.

## Task 2: Rotary embeddings in two dimensions

RoPE rotates a vector by an angle proportional to its position. Write `rotate(v, position, theta=0.3)` that rotates the 2D vector `v` by the angle `position * theta` (counter-clockwise). Then, for the fixed vectors `q` and `k` below,
compute `score(m, n) = rotate(q, m) @ rotate(k, n)` and store in `same_gap` whether `score(3, 1)`, `score(10, 8)` and `score(7, 5)` (all a gap of 2) are equal, and in `different_gap` whether `score(3, 1)` differs from `score(3, 0)`.

In [4]:
q, k = np.array([1.0, 0.5]), np.array([0.3, 1.0])


def rotate(v, position, theta=0.3):
    a = position * theta
    return np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]]) @ v


def score(m, n):
    return rotate(q, m) @ rotate(k, n)


same_gap = bool(np.isclose(score(3, 1), score(10, 8)) and np.isclose(score(3, 1), score(7, 5)))
different_gap = bool(not np.isclose(score(3, 1), score(3, 0)))

In [5]:
check("rotating by position 0 changes nothing", lambda: close(rotate(np.array([1.0, 2.0]), 0), [1.0, 2.0], 1e-12))
check("rotation keeps the length of a vector", lambda: close(np.linalg.norm(rotate(q, 5)), np.linalg.norm(q), 1e-12))
check("equal gaps give equal scores", lambda: same_gap is True)
check("different gaps give different scores", lambda: different_gap is True)
summary()

PASS  rotating by position 0 changes nothing
PASS  rotation keeps the length of a vector
PASS  equal gaps give equal scores
PASS  different gaps give different scores

4 of 4 checks passed.  Well done.


**Why it works.** Rotating by m*theta and n*theta turns the dot product into a function of (m - n)*theta alone, because the two rotations combine into one. So the score knows how far apart two tokens are without storing absolute positions.